# 05 - Time travel e auditoria

## Objetivo

Consultar versões históricas da tabela Iceberg.

## Valor para a PRODEMGE

Responder perguntas como: “Como estava a base antes da correção?” ou “Qual era o estado dos pagamentos em determinada data?”

## Como usar

1. Substitua os placeholders (`<datahub_link>`, `<usuário>` e `<senha>`) no primeiro bloco de código.
2. Execute a célula de configuração Livy.
3. Execute as células da demo.
4. No final, encerre a sessão Livy.

> Este notebook não executa Spark localmente. Todo código Spark SQL/PySpark é submetido ao Livy3 via REST API.

In [ ]:
import requests
import time
import json
import urllib3

# Remove warnings de certificado SSL self-signed
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# =====================================================================
# CONFIGURAÇÕES DO LIVY3
# =====================================================================

LIVY_URL = "<datahub_link>"
USERNAME = "<usuário>"
PASSWORD = "<senha>"


BASE_LIVY_URL = LIVY_URL.rstrip("/")

# =====================================================================
# CONFIGURAÇÕES SPARK
# =====================================================================

SESSION_CONF = {
    "spark.app.name": "PRODEMGE_Iceberg_Demo_Livy3",

    "spark.executor.memory": "4g",
    "spark.executor.cores": "2",
    "spark.executor.instances": "2",

    "spark.driver.memory": "2g",

    "spark.sql.catalog.iceberg_prod":
        "org.apache.iceberg.spark.SparkCatalog",

    "spark.sql.catalog.iceberg_prod.type":
        "hive",

    "spark.sql.extensions":
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions"
}

# =====================================================================
# CLIENTE HTTP
# =====================================================================

http = requests.Session()

http.auth = (USERNAME, PASSWORD)
http.verify = False

HEADERS = {
    "Content-Type": "application/json"
}

# =====================================================================
# AUXILIARES
# =====================================================================

def pretty_json(obj):
    print(
        json.dumps(
            obj,
            indent=2,
            ensure_ascii=False
        )
    )

# =====================================================================
# CRIAÇÃO DA SESSÃO LIVY
# =====================================================================

def create_livy_session(
    kind="pyspark",
    conf=None,
    timeout_seconds=600
):

    payload = {
        "kind": kind,
        "conf": conf or SESSION_CONF
    }

    response = http.post(
        f"{BASE_LIVY_URL}/sessions",
        headers=HEADERS,
        data=json.dumps(payload),
        verify=False
    )

    response.raise_for_status()

    session_id = response.json()["id"]

    print(f"Sessão Livy criada: {session_id}")

    start = time.time()

    while True:

        response = http.get(
            f"{BASE_LIVY_URL}/sessions/{session_id}",
            verify=False
        )

        response.raise_for_status()

        payload = response.json()

        state = payload.get("state")

        print(f"Estado da sessão: {state}")

        if state == "idle":

            print(
                "Sessão Livy pronta para receber statements."
            )

            return session_id

        if state in ["dead", "error", "killed"]:

            pretty_json(payload)

            raise RuntimeError(
                f"Falha ao criar sessão Livy. Estado: {state}"
            )

        if time.time() - start > timeout_seconds:

            raise TimeoutError(
                "Timeout aguardando sessão Livy ficar idle."
            )

        time.sleep(5)

# =====================================================================
# EXECUÇÃO DE STATEMENTS
# =====================================================================

def submit_statement(
    session_id,
    code,
    kind="pyspark",
    timeout_seconds=900
):

    payload = {
        "code": code,
        "kind": kind
    }

    response = http.post(
        f"{BASE_LIVY_URL}/sessions/{session_id}/statements",
        headers=HEADERS,
        data=json.dumps(payload),
        verify=False
    )

    response.raise_for_status()

    statement_id = response.json()["id"]

    print(
        f"Statement submetido: {statement_id}"
    )

    start = time.time()

    while True:

        response = http.get(
            f"{BASE_LIVY_URL}/sessions/{session_id}/statements/{statement_id}",
            verify=False
        )

        response.raise_for_status()

        result = response.json()

        state = result.get("state")

        print(
            f"Estado do statement: {state}"
        )

        if state == "available":

            output = result.get(
                "output",
                {}
            )

            pretty_json(output)

            return output

        if state in [
            "error",
            "cancelling",
            "cancelled"
        ]:

            pretty_json(result)

            raise RuntimeError(
                f"Statement falhou. Estado: {state}"
            )

        if time.time() - start > timeout_seconds:

            raise TimeoutError(
                "Timeout aguardando statement finalizar."
            )

        time.sleep(3)

# =====================================================================
# ENCERRAMENTO DA SESSÃO
# =====================================================================

def close_livy_session(session_id):

    response = http.delete(
        f"{BASE_LIVY_URL}/sessions/{session_id}",
        verify=False
    )

    if response.status_code in [
        200,
        202,
        204
    ]:

        print(
            f"Sessão Livy encerrada: {session_id}"
        )

    else:

        print(
            f"Não foi possível encerrar a sessão {session_id}"
        )

        print(response.text)

# =====================================================================
# TESTE
# =====================================================================

livy_session_id = create_livy_session()

print(
    f"Sessão criada com sucesso: {livy_session_id}"
)

Sessão Livy criada: 12
Estado da sessão: starting
Estado da sessão: starting
Estado da sessão: starting
Estado da sessão: starting
Estado da sessão: starting
Estado da sessão: starting
Estado da sessão: starting
Estado da sessão: idle
Sessão Livy pronta para receber statements.


In [ ]:
# Normaliza o username para usar no nome da tabela sem caracteres inválidos.
table_suffix = "".join(ch if ch.isalnum() or ch == "_" else "_" for ch in USERNAME).strip("_")
table_name = f"beneficios_{table_suffix}"
particionado_table = f"beneficios_particionados_{table_suffix}"
staging_table = f"beneficios_staging_{table_suffix}"
csv_table = f"beneficios_csv_{table_suffix}"

spark_code = f'''
# =====================================================================
# TIME TRAVEL E AUDITORIA
# =====================================================================

snapshots_df = spark.sql("""
SELECT snapshot_id, committed_at, operation
FROM governo_mg.{table_name}.snapshots
ORDER BY committed_at DESC
""")

snapshots_df.show(truncate=False)

snapshots = snapshots_df.collect()

if len(snapshots) >= 1:

    # Seleciona o snapshot mais antigo disponível para demonstrar time travel.
    snapshot_id = snapshots[-1]["snapshot_id"]

    print(f"Snapshot selecionado para time travel: {{snapshot_id}}")

    spark.sql(f"""
    SELECT *
    FROM governo_mg.{table_name} VERSION AS OF {{snapshot_id}}
    """).show(truncate=False)

else:
    print("Nenhum snapshot encontrado.")
'''

submit_statement(livy_session_id, spark_code)

Statement submetido: 0
Estado do statement: waiting
Estado do statement: running
Estado do statement: running
Estado do statement: running
Estado do statement: running
Estado do statement: available
{
  "status": "ok",
  "execution_count": 0,
  "data": {
    "text/plain": "+-------------------+-----------------------+---------+\n|snapshot_id        |committed_at           |operation|\n+-------------------+-----------------------+---------+\n|392287009861455930 |2026-05-28 18:38:49.502|overwrite|\n|2361585162724279169|2026-05-28 18:38:47.023|overwrite|\n|119708453340927850 |2026-05-28 18:00:01.362|append   |\n+-------------------+-----------------------+---------+\n\nSnapshot selecionado para time travel: 119708453340927850\n+----------+-----------+---------------+--------------+--------+\n|id_cidadao|nome       |valor_beneficio|data_pagamento|status  |\n+----------+-----------+---------------+--------------+--------+\n|1         |Maria Silva|450.0          |2025-01-15    |ATIVO   |\n|2

{'status': 'ok',
 'execution_count': 0,
 'data': {'text/plain': '+-------------------+-----------------------+---------+\n|snapshot_id        |committed_at           |operation|\n+-------------------+-----------------------+---------+\n|392287009861455930 |2026-05-28 18:38:49.502|overwrite|\n|2361585162724279169|2026-05-28 18:38:47.023|overwrite|\n|119708453340927850 |2026-05-28 18:00:01.362|append   |\n+-------------------+-----------------------+---------+\n\nSnapshot selecionado para time travel: 119708453340927850\n+----------+-----------+---------------+--------------+--------+\n|id_cidadao|nome       |valor_beneficio|data_pagamento|status  |\n+----------+-----------+---------------+--------------+--------+\n|1         |Maria Silva|450.0          |2025-01-15    |ATIVO   |\n|2         |João Souza |600.0          |2025-01-20    |ATIVO   |\n|3         |Ana Lima   |350.0          |2025-02-10    |SUSPENSO|\n+----------+-----------+---------------+--------------+--------+'}}

## Exemplo no Hue / Impala

```sql
SELECT *
FROM governo_mg.beneficios
FOR SYSTEM_VERSION AS OF <snapshot_id>;
```

> A sintaxe exata pode variar conforme versão/runtime. Use a opção suportada pelo seu engine no ambiente Cloudera.

In [3]:
# Encerre a sessão ao final do notebook.
close_livy_session(livy_session_id)

Sessão Livy encerrada: 12
